# AI Replace Inpaint Smoke Runner

Standalone Pokecut-style AI Replace flow under `inpaint/`.

This notebook mirrors the main SD3.5/Add-it runner shape: install, clone/update, imports, runtime check, optional HF login, smoke run, metrics, previews, export.


## 1. Install Dependencies

In [ ]:
!pip install -q "diffusers>=0.30.0,<1.0.0" "transformers>=4.40.0" "accelerate>=0.30.0" safetensors ultralytics huggingface_hub opencv-python pillow numpy


## 2. Clone Or Update Repo

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/BDT-17/VIN.git"
REPO_DIR = Path("/kaggle/working/VIN")

if REPO_DIR.exists():
    %cd /kaggle/working/VIN
    !git fetch origin main
    !git pull --ff-only origin main
else:
    !git clone {REPO_URL} {REPO_DIR}
    %cd /kaggle/working/VIN

PROJECT_DIR = REPO_DIR if (REPO_DIR / "inpaint").exists() else Path.cwd()
print("PROJECT_DIR:", PROJECT_DIR)


## 3. Imports

In [ ]:
import os
import sys
import json
import shutil
from pathlib import Path

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

PROJECT_DIR = Path("/kaggle/working/VIN") if Path("/kaggle/working/VIN/inpaint").exists() else Path.cwd()
sys.path = [str(PROJECT_DIR)] + [path for path in sys.path if path != str(PROJECT_DIR)]
%cd {PROJECT_DIR}

from inpaint.config import DEFAULT_CONFIG, AIReplaceConfig
from inpaint.smoke_runner import run_smoke, git_pull_ff_only, list_source_images

print("Loaded inpaint flow:", DEFAULT_CONFIG.AI_REPLACE_FLOW)
print("Model:", DEFAULT_CONFIG.MODEL_ID)


## 4. Runtime Check

In [ ]:
import torch

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu count:", torch.cuda.device_count())
    for idx in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(idx)
        print(idx, props.name, round(props.total_memory / 1024**3, 2), "GB")


## 5. Hugging Face Login

In [ ]:
import os
from huggingface_hub import login

hf_token = os.environ.get("HF_TOKEN")
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = hf_token or UserSecretsClient().get_secret("HF_TOKEN")
except Exception as exc:
    print("No Kaggle HF_TOKEN secret found:", type(exc).__name__)

if hf_token:
    login(token=hf_token)
    print("Logged in to Hugging Face")
else:
    print("No HF token provided; public models only")


## 6. Smoke Config

In [ ]:
# Change these for your Kaggle dataset mount.
INPUT_DIR = Path("/kaggle/input/your-dataset")
OUTPUT_DIR = Path("/kaggle/working/ai_replace_smoke")
NUM_IMAGES = 20
SEED = 42

# Smoke modes:
# - DRY_RUN=True validates mask/restore/manifest without loading SD model.
# - DRY_RUN=False loads SD2 inpainting and generates images.
DRY_RUN = False
USE_YOLO = True

print("INPUT_DIR exists:", INPUT_DIR.exists(), INPUT_DIR)
if INPUT_DIR.exists():
    sample_paths = list_source_images(INPUT_DIR, limit=5)
    print("sample images:", [p.name for p in sample_paths])
print("OUTPUT_DIR:", OUTPUT_DIR)


## 7. Optional Dry-Run Wiring Test

Use this before loading the model if you only want to verify repo import, masks, hard restore, manifest, metrics and previews.


In [ ]:
if INPUT_DIR.exists():
    dry_rows, dry_summary = run_smoke(
        input_dir=INPUT_DIR,
        output_dir=Path("/kaggle/working/ai_replace_dry_run"),
        num_images=min(3, NUM_IMAGES),
        seed=SEED,
        load_model=False,
        use_yolo=False,
    )
    dry_summary
else:
    print("Set INPUT_DIR before running dry-run.")


## 8. Full Model Smoke Test

In [ ]:
rows, summary = run_smoke(
    input_dir=INPUT_DIR,
    output_dir=OUTPUT_DIR,
    num_images=NUM_IMAGES,
    seed=SEED,
    load_model=not DRY_RUN,
    use_yolo=USE_YOLO,
)
summary


## 9. Metrics Summary

In [ ]:
summary_path = OUTPUT_DIR / "metrics" / "metrics_summary.json"
summary = json.load(open(summary_path, encoding="utf-8"))
summary


## 10. Manifest Preview

In [ ]:
import pandas as pd

manifest_csv = OUTPUT_DIR / "manifest.csv"
df = pd.read_csv(manifest_csv)
print("rows:", len(df))
display_cols = [
    "accepted", "reject_reason", "outside_mask_diff", "object_mask_inside_ratio",
    "opacity_score", "background_preservation_score", "ai_replace_quality_score",
    "source_path", "output_path",
]
df[[col for col in display_cols if col in df.columns]].head(20)


## 11. Preview Gallery

In [ ]:
import math
import matplotlib.pyplot as plt
from PIL import Image

preview_dir = OUTPUT_DIR / "previews"
harmonized = sorted(preview_dir.glob("*_harmonized.png"))[:6]
if not harmonized:
    print("No previews found")
else:
    cols = 3
    rows_n = math.ceil(len(harmonized) / cols)
    plt.figure(figsize=(5 * cols, 5 * rows_n))
    for i, path in enumerate(harmonized, 1):
        plt.subplot(rows_n, cols, i)
        plt.imshow(Image.open(path))
        plt.title(path.name[:48])
        plt.axis("off")
    plt.tight_layout()


## 12. Outside-Mask Diff Preview

These should be nearly black when hard restore is working.


In [ ]:
diffs = sorted((OUTPUT_DIR / "previews").glob("*_diff_outside_mask.png"))[:6]
if not diffs:
    print("No diff previews found")
else:
    cols = 3
    rows_n = math.ceil(len(diffs) / cols)
    plt.figure(figsize=(5 * cols, 5 * rows_n))
    for i, path in enumerate(diffs, 1):
        plt.subplot(rows_n, cols, i)
        plt.imshow(Image.open(path))
        plt.title(path.name[:48])
        plt.axis("off")
    plt.tight_layout()


## 13. Export Outputs

In [ ]:
zip_base = Path('/kaggle/working') / OUTPUT_DIR.name
zip_path = shutil.make_archive(str(zip_base), 'zip', root_dir=OUTPUT_DIR)
print('Saved export:', zip_path)
